# Titanic Survival Prediction — Model Training & Comparison

This notebook trains and compares multiple classification models to predict
Titanic passenger survival, using the preprocessed data from `src/preprocessing.py`.

**Goal:** train several models, compare their performance using cross-validation,
and identify the strongest candidate(s) for further tuning.

**Baseline to beat:** 61.6% accuracy (always predicting "did not survive").

In [ ]:
import sys
sys.path.append('../src')
from preprocessing import full_pipeline

X_train, X_val, y_train, y_val = full_pipeline('../data/train.csv')
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(712, 14) (179, 14) (712,) (179,)


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_val)

In [9]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_val, predictions)
print(accuracy)

0.8156424581005587


In [12]:
import pandas as pd

coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

              Feature  Coefficient
8              Master     2.408691
3         Sex_encoded     2.158674
10                Mrs     1.515419
13           HasCabin     0.577723
9                Miss     0.435669
6                   C     0.416222
7                   Q     0.263078
11               Rare     0.187106
12               Fare     0.003521
4          Age_filled    -0.017176
5   Age_Group_encoded    -0.161133
2             isAlone    -0.269028
1          FamilySize    -0.466629
0              Pclass    -0.656286


# Logistic Regression

**What it is:** Despite the name, Logistic Regression is a *classification* model (predicts categories), not a regression model that predicts continuous numbers.

**How it works:**
1. Each feature (Pclass, Sex_encoded, Fare, etc.) is multiplied by a learned weight, and all these weighted features are summed together (plus an intercept) — producing a "raw score" that can be any number, positive or negative.
2. That raw score is passed through the **sigmoid function**, an S-shaped curve that squishes any input into a probability between 0 and 1:
   - Large positive raw score → probability close to 1 (likely survived)
   - Large negative raw score → probability close to 0 (likely did not survive)
   - Raw score of 0 → probability of exactly 0.5 (model is unsure)
3. If the resulting probability is above 0.5, the model predicts "survived" (1); otherwise, "did not survive" (0)

**Why it's a good baseline model:**
- Simple and fast to train
- Interpretable — the learned weights show which features push predictions toward survival vs. death, and by how much
- Gives a solid reference point to compare more complex models against later

**Training results:**
- `LogisticRegression(max_iter=1000)` — increased `max_iter` from the default 100 to resolve a convergence warning, likely caused by features being on very different scales (e.g., Fare ranges ~0-500, Sex_encoded is only 0 or 1)
- **Validation accuracy: 81.6%** — a strong improvement over the 61.6% baseline (always predicting "did not survive"), the accuracy means the model correctly predicted survived.death for about 81.6% of the 179 validation passengers
- Note: this accuracy comes from a single train/validation split (`random_state=42`); cross-validation will be used later for a more reliable performance estimate across multiple splits

**Possible future improvement:** apply feature scaling (e.g., StandardScaler) to put all features on a comparable range, which may improve convergence and potentially model performance for Logistic Regression specifically (tree-based models like Random Forest are generally unaffected by feature scale).

**Coefficients used by Logistic Regression**

1. Strongest positive coefficients (push towards survival)
   - `Master` : 2.41, being a young boy strongly increases predicted survival probability
   - `Sex_encoded` : 2.16, being female strongly inceases predicred survival probability
   - `Mrs` : 1.52, being a Married woman stronlgy increases predicted survival probability
   - `HasCabin` : 0.58 having a recorded cabin icreases predicted survival probabilty
2. Strongest negative coefficients (push towards death)
   - `Pclass` : -0.66, as class number increases 1-> 2 -> 3, survival probability drops
   - `FamilySize` : - 0.47, limitation of linear coefficient, since in EDA we saw that it is a curve (small famileis did better, only very large families did worse)
   - `isAlone` : -0.27, being alone decreases survival probability
3. Near-zero coefficients
- `Fare` : 0.0035, nearly zero despite Fare showing a 0.26 correlation with survival in EDA. Fare was mostly a proxy for class/wealth, and `Pclass` (which already has a strong -0.66 coefficient) captures that same signal more directly — once Pclass is accounted for, Fare has little independent information left to contribute
   - `Age_filled` : -0.017, nearly zero despite the "children first" pattern found in EDA. Most of the strong age-related signal (specifically, young boys) is already captured by the `Master` feature (2.41). The remaining age-survival relationship, once young boys are excluded, is not a clean, single-direction trend across the rest of the age range — Logistic Regression can only fit one consistent slope per feature, so a relationship that's strong in one narrow sub-range but weak/inconsistent elsewhere ends up averaging out to a coefficient close to zero
   - `Age_Group_encoded` : -0.16, similarly small — likely for the same reason as Age_filled, since it's a more coarse-grained version of the same information